## Project Checklist

| No. | Requirement                                                         | Present (Yes/No) | Comments                      |
|-----|---------------------------------------------------------------------|------------------|-------------------------------|
|  1  | Data loaded as per instructions                                     | Yes              |                               |
|  2  | RAG approach using HuggingFace/Groq LLM                             | Yes              |                               |
|  3  | Vector DB (e.g. FAISS) used for retrieval                           | Yes              |                               |
|  4  | Embeddings computed                                                 | Yes              |                               |
|  5  | Query rewriting using HuggingFace or Gemma LLM                      | Yes              |                               |
|  6  | Model blocks out-of-context responses                               | Yes              |                               |
|  7  | Model tuning performed                                              | Yes              |                               |
|  8  | Evaluation: ROUGE-L and BERT-F1                                     | Yes              |                               |
|  9  | Retrieval eval: MAP and MRR                                         | Yes              |                               |



In [1]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
from tqdm import tqdm
# === Imports ===
import pandas as pd
from langchain.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import numpy as np
import ast
import evaluate
import bert_score

In [3]:
# !pip install transformers faiss-cpu tqdm sentence-transformers langchain langchain-huggingface datasets
# !pip install evaluate bert-score
# !pip install rouge_score


In [4]:
base_folder = "rag_with_rewrite_query_files/"
import os

if not os.path.exists(base_folder):
    os.makedirs(base_folder)

In [5]:
df_psg = pd.read_parquet("hf://datasets/rag-datasets/rag-mini-bioasq/data/passages.parquet/part.0.parquet")
df_test = pd.read_parquet("hf://datasets/rag-datasets/rag-mini-bioasq/data/test.parquet/part.0.parquet")

df_psg = df_psg.reset_index()
df_test = df_test.reset_index()

df_psg = df_psg.rename(columns={'id': 'doc_id'})
df_test = df_test.rename(columns={'id': 'doc_id'})


In [6]:
df_psg.to_csv(base_folder + "df_psg.csv", index=False)
df_test.to_csv(base_folder + "df_test.csv", index=False)

In [7]:
# Load DataFrames
df_passages = pd.read_csv(base_folder + 'df_psg.csv')   # update path as needed
df_test = pd.read_csv(base_folder + 'df_test.csv')      # update path as needed

# Clean: Remove rows where passage is null or duplicated
df_passages = df_passages.dropna(subset=['passage'])
df_passages = df_passages.drop_duplicates(subset='passage')
df_passages = df_passages.reset_index(drop=True)

# Clean test DataFrame
df_test = df_test.dropna(subset=['question', 'answer'])
df_test = df_test.drop_duplicates(subset='question')
df_test = df_test.reset_index(drop=True)

print("Passages:", df_passages.shape)
print("Test Q&A:", df_test.shape)

# Extract both chunks and doc_ids (removing nulls ensures alignment)
chunks = df_passages['passage'].tolist()
doc_ids = df_passages['doc_id'].tolist()


Passages: (27974, 2)
Test Q&A: (4719, 4)


In [8]:
df_passages.head(5)

,doc_id,passage
0,9797,New data on viruses isolated from patients wit...
1,11906,We describe an improved method for detecting d...
2,16083,We have studied the effects of curare on respo...
3,23188,Kinetic and electrophoretic properties of 230-...
4,23469,Male Wistar specific-pathogen-free rats aged 2...


In [9]:
df_test.head(5)

,doc_id,question,answer,relevant_passage_ids
0,0,Is Hirschsprung disease a mendelian or a multi...,"Coding sequence mutations in RET, GDNF, EDNRB,...","[20598273, 6650562, 15829955, 15617541, 230011..."
1,1,List signaling molecules (ligands) that intera...,The 7 known EGFR ligands are: epidermal growt...,"[23821377, 24323361, 23382875, 22247333, 23787..."
2,2,Is the protein Papilin secreted?,"Yes, papilin is a secreted protein","[21784067, 19297413, 15094122, 7515725, 332004..."
3,3,Are long non coding RNAs spliced?,Long non coding RNAs appear to be spliced thro...,"[22955974, 21622663, 22707570, 22955988, 24285..."
4,4,Is RANKL secreted from the cells?,Receptor activator of nuclear factor κB ligand...,"[22867712, 23827649, 21618594, 23835909, 24265..."


In [10]:
# from langchain_huggingface import HuggingFaceEmbeddings
# from langchain.vectorstores import FAISS
# from langchain.docstore.document import Document
# from tqdm import tqdm

# embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# print("Generating embeddings...")
# embeddings = [embedding_model.embed_documents([text])[0] for text in tqdm(chunks, desc="Embedding Chunks")]


In [11]:
# print("Building LangChain documents with metadata...")
# docs = [
#     Document(page_content=text, metadata={"doc_id": doc_id})
#     for text, doc_id in zip(chunks, doc_ids)
# ]

# print("Building FAISS index...")
# vector_db = FAISS.from_documents(docs, embedding_model)
# print("FAISS index created.")

In [12]:
# # Save the FAISS index and metadata
# vector_db.save_local("faiss_index_folder")
# print("FAISS index saved to 'faiss_index_folder'.")


In [13]:
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Load from local folder
vector_db = FAISS.load_local(
    "faiss_index_folder",
    embedding_model,
    allow_dangerous_deserialization=True
)
print("FAISS index loaded.")


FAISS index loaded.


In [14]:
model_dir = "saved_models/distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(model_dir)
tokenizer.pad_token = tokenizer.eos_token   # Fix pad_token
tokenizer.padding_side = "left"             # Fix padding side
model = AutoModelForCausalLM.from_pretrained(model_dir)

def rewrite_queries(queries, batch_size=32):
    rewritten_outputs = []
    for i in tqdm(range(0, len(queries), batch_size), desc="Rewriting Queries"):
        batch = queries[i:i+batch_size]
        # Prompt engineering for all in batch
        prompts = [f"Rewrite this biomedical question to be more domain-specific: {q}" for q in batch]
        inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True)
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=100,
                pad_token_id=tokenizer.eos_token_id
            )
        batch_rewrites = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        rewritten_outputs.extend(batch_rewrites)
    return rewritten_outputs


In [15]:
def retrieve_top_k(query, k=10):
    docs = vector_db.similarity_search(query, k=k)
    # Extract the doc_id from metadata for each Document
    return [doc.metadata['doc_id'] for doc in docs]


In [16]:
def is_in_context(query):
    keywords = ["gene", "protein", "cell", "disease", "biomedical", "treatment", "mutation", "symptom", "signaling", "pathway"]
    return any(word in query.lower() for word in keywords)


In [17]:
k=10

In [18]:
k = 10

retrieved_texts = []
retrieved_ids = []
top_passages_list = []

queries = df_test['question'].tolist()
rewritten_queries = rewrite_queries(queries, batch_size=32)

for i in tqdm(range(len(df_test)), desc="Retrieval"):
    query = queries[i]
    if not is_in_context(query):
        retrieved_texts.append("Sorry, I can only answer biomedical science questions.")
        retrieved_ids.append([-1]*k)
        top_passages_list.append([])
        continue

    rewritten = rewritten_queries[i]
    top_docs = vector_db.similarity_search(rewritten, k=k)
    top_passages = [doc.page_content for doc in top_docs]
    top_ids = [doc.metadata.get('doc_id', -1) for doc in top_docs] if top_docs and hasattr(top_docs[0], 'metadata') else [-1]*k

    retrieved_texts.append(" ".join(top_passages))
    retrieved_ids.append(top_ids)
    top_passages_list.append(top_passages)

gold_answers = df_test['answer'].tolist()
relevant_ids = df_test['relevant_passage_ids'].tolist()


Retrieval: 100%|██████████| 4719/4719 [00:11<00:00, 397.86it/s]


In [19]:
def generate_rag_answer(query, retrieved_passages, tokenizer, model, max_context=1500, max_new_tokens=128):
    """
    Generates a final answer by providing the query and the retrieved passages as context.
    """
    context = " ".join(retrieved_passages)[:max_context]
    prompt = (
        f"Context:\n{context}\n\n"
        f"Question: {query}\n"
        f"Answer:"
    )
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=tokenizer.model_max_length)
    input_ids = inputs['input_ids']
    with torch.no_grad():
        output_ids = model.generate(
            input_ids=input_ids,
            attention_mask=inputs['attention_mask'],
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.eos_token_id
        )
    output = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    # Only return the generated answer (strip out the prompt)
    return output[len(prompt):].strip()


In [20]:
# Batch Generate RAG Answers ===
final_rag_answers = []
for i in tqdm(range(len(df_test)), desc="RAG Generation"):
    query = queries[i]
    if not is_in_context(query):
        final_rag_answers.append("Sorry, I can only answer biomedical science questions.")
        continue
    answer = generate_rag_answer(query, top_passages_list[i], tokenizer, model)
    final_rag_answers.append(answer)

RAG Generation: 100%|██████████| 4719/4719 [20:18<00:00,  3.87it/s]


In [21]:
print("Final RAG answers:", final_rag_answers[:2])

Final RAG answers: ["Yes.\nThe most common cause of Hirschsprung's disease is the\nuniformity of the \nuniformity of the \nuniformity of the \nuniformity of the \nuniformity of the \nuniformity of the \nuniformity of the \nuniformity of the \nuniformity of the \nuniformity of the \nuniformity of the \nuniformity of the \nuniformity of the \nuniformity of the \nuniformity of the \nuniformity of the", 'The most common ligands are \nEGF>B-EGF>TGF-α>BTC>EPR>EPG>AR.\nQuestion: How do you define the binding of \nEGF>B-EGF>TGF-α>BTC>EPR>EPG>AR.\nQuestion: How do you define the binding of \nEGF>B-EGF>TGF-α>BTC>EPR>EPG>AR.\nQuestion: How do you define the binding of \nEGF>B-EGF>TGF-α>BTC>']


In [22]:
import pickle
# Save retrieval results
pd.DataFrame({
    "question": df_test['question'],
    "retrieved_texts": retrieved_texts,
    "retrieved_ids": retrieved_ids,
    "final_rag_answers": final_rag_answers
}).to_csv(base_folder + "retrieval_results.csv", index=False)
# Save for reproducibility
with open(base_folder + "gold_answers.pkl", "wb") as f:
    pickle.dump(gold_answers, f)
with open(base_folder + "relevant_ids.pkl", "wb") as f:
    pickle.dump(relevant_ids, f)


In [23]:
import pandas as pd
import pickle
import ast

base_folder = "rag_with_rewrite_query_files/"

# Read CSV with retrieval results
df_results = pd.read_csv(base_folder + "retrieval_results.csv")
retrieved_texts = df_results["retrieved_texts"].tolist()
retrieved_ids = [ast.literal_eval(x) for x in df_results["retrieved_ids"].tolist()]  # Convert string-lists to lists if needed
final_rag_answers = df_results["final_rag_answers"].tolist()
questions = df_results["question"].tolist()

# Read pickled lists
with open(base_folder + "gold_answers.pkl", "rb") as f:
    gold_answers = pickle.load(f)
with open(base_folder + "relevant_ids.pkl", "rb") as f:
    relevant_ids = pickle.load(f)

# Quick check: print first 2 entries of each loaded item
print("Questions:", questions[:2])
print("Retrieved IDs:", retrieved_ids[:2])
print("Final RAG answers:", final_rag_answers[:2])
print("Gold answers:", gold_answers[:2])
print("Relevant IDs:", relevant_ids[:2])


Questions: ['Is Hirschsprung disease a mendelian or a multifactorial disorder?', 'List signaling molecules (ligands) that interact with the receptor EGFR?']
Retrieved IDs: [[23001136, 1785632, 17965226, 22584707, 9727738, 12878302, 9745455, 7605558, 9571278, 15617541], [23382875, 22829865, 23637683, 34667080, 27426127, 21400356, 29680500, 31770593, 23399900, 19048033]]
Final RAG answers: ["Yes.\nThe most common cause of Hirschsprung's disease is the\nuniformity of the \nuniformity of the \nuniformity of the \nuniformity of the \nuniformity of the \nuniformity of the \nuniformity of the \nuniformity of the \nuniformity of the \nuniformity of the \nuniformity of the \nuniformity of the \nuniformity of the \nuniformity of the \nuniformity of the \nuniformity of the", 'The most common ligands are \nEGF>B-EGF>TGF-α>BTC>EPR>EPG>AR.\nQuestion: How do you define the binding of \nEGF>B-EGF>TGF-α>BTC>EPR>EPG>AR.\nQuestion: How do you define the binding of \nEGF>B-EGF>TGF-α>BTC>EPR>EPG>AR.\nQuest

In [24]:
import ast
import evaluate
import numpy as np
import bert_score
from tqdm import tqdm
import torch

# Device selection for bert_score
device = "cuda" if torch.cuda.is_available() else ("mps" if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available() else "cpu")

# Load metrics
rouge = evaluate.load('rouge')
bertscore = evaluate.load('bertscore')

def mean_average_precision(retrieved_ids, relevant_ids):
    """
    Compute MAP between retrieved and relevant IDs.
    """
    avg_precisions = []
    relevant_ids = [ast.literal_eval(rels) if isinstance(rels, str) else rels for rels in relevant_ids]
    for retrieved, relevant in tqdm(zip(retrieved_ids, relevant_ids), total=len(retrieved_ids), desc="Computing MAP"):
        score = 0.0
        hits = 0
        for i, pid in enumerate(retrieved):
            if pid in relevant:
                hits += 1
                score += hits / (i + 1)
        if relevant:
            avg_precisions.append(score / len(relevant))
    return sum(avg_precisions) / len(avg_precisions)

def mean_reciprocal_rank(retrieved_ids, relevant_ids):
    """
    Compute MRR between retrieved and relevant IDs.
    """
    rr_scores = []
    relevant_ids = [ast.literal_eval(rels) if isinstance(rels, str) else rels for rels in relevant_ids]
    for retrieved, relevant in tqdm(zip(retrieved_ids, relevant_ids), total=len(retrieved_ids), desc="Computing MRR"):
        for i, pid in enumerate(retrieved):
            if pid in relevant:
                rr_scores.append(1 / (i + 1))
                break
        else:
            rr_scores.append(0.0)
    return sum(rr_scores) / len(rr_scores)

def compute_generation_scores(gold_answers, retrieved_passages):
    """
    Compute ROUGE-L and BERT-F1 between generated and gold answers.
    """
    rouge_l = rouge.compute(predictions=retrieved_passages, references=gold_answers)
    bert_f1 = bert_score.score(
        cands=retrieved_passages,
        refs=gold_answers,
        lang="en",
        model_type="distilbert-base-uncased",
        verbose=False,
        batch_size=128,
        device=device
    )[2]
    return {
        "ROUGE-L": rouge_l["rougeL"],
        "BERT-F1": float(bert_f1.mean())
    }


In [25]:
from tqdm import tqdm

map_score = mean_average_precision(retrieved_ids, relevant_ids)
mrr_score = mean_reciprocal_rank(retrieved_ids, relevant_ids)
print(f"\nRetrieval Scores:\nMAP@{k}: {map_score:.4f}\nMRR@{k}: {mrr_score:.4f}")

Computing MRR: 100%|██████████| 4719/4719 [00:00<00:00, 2326936.35it/s]


Retrieval Scores:
MAP@10: 0.0976
MRR@10: 0.2215


In [26]:
print("retrieved_ids sample:", retrieved_ids[0])
print("relevant_ids sample:", relevant_ids[0])

retrieved_ids sample: [23001136, 1785632, 17965226, 22584707, 9727738, 12878302, 9745455, 7605558, 9571278, 15617541]
relevant_ids sample: [20598273, 6650562, 15829955, 15617541, 23001136, 8896569, 21995290, 12239580, 15858239]


In [27]:
gen_scores_baseline = compute_generation_scores(gold_answers, retrieved_texts)

print("\nBaseline Generation Scores:")
print(gen_scores_baseline)

print("Retrieval-Only - ROUGE-L:", gen_scores_baseline["ROUGE-L"])
print("Retrieval-Only - BERT-F1:", gen_scores_baseline["BERT-F1"])

gen_scores_rag = compute_generation_scores(gold_answers, final_rag_answers)

print("\nRAG Generation Scores:")
print(gen_scores_rag)

print("RAG - ROUGE-L:", gen_scores_rag["ROUGE-L"])
print("RAG - BERT-F1:", gen_scores_rag["BERT-F1"])

eval_summary = pd.DataFrame({
    "Metric": ["BASELINE_ROUGE-L", "BASELINE_BERT-F1", "RAG_ROUGE-L", "RAG_BERT-F1",f"MAP@{k}", f"MRR@{k}"],
    "Score": [gen_scores_baseline["ROUGE-L"], gen_scores_baseline["BERT-F1"], gen_scores_rag["ROUGE-L"], gen_scores_rag["BERT-F1"], map_score, mrr_score]
})


Baseline Generation Scores:
{'ROUGE-L': 0.011064887220118464, 'BERT-F1': 0.7011651396751404}
Retrieval-Only - ROUGE-L: 0.011064887220118464
Retrieval-Only - BERT-F1: 0.7011651396751404

RAG Generation Scores:
{'ROUGE-L': 0.038315823582424795, 'BERT-F1': 0.6892945170402527}
RAG - ROUGE-L: 0.038315823582424795
RAG - BERT-F1: 0.6892945170402527


In [28]:
chunks = df_passages['passage'].tolist()
doc_ids = df_passages['doc_id'].tolist()

# Save for reproducibility
pd.DataFrame({"doc_id": doc_ids, "passage": chunks}).to_csv(base_folder + "chunks_and_ids.csv", index=False)


In [29]:
base_folder = "rag_with_rewrite_query_files/"

# Read the CSV
df_chunks = pd.read_csv(base_folder + "chunks_and_ids.csv")

# Convert to lists
doc_ids = df_chunks["doc_id"].tolist()
chunks = df_chunks["passage"].tolist()

# Quick check
print("First 2 doc_ids:", doc_ids[:2])
print("First 2 passages:", chunks[:2])


First 2 doc_ids: [9797, 11906]
First 2 passages: ['New data on viruses isolated from patients with subacute thyroiditis de Quervain \nare reported. Characteristic morphological, cytological, some physico-chemical \nand biological features of the isolated viruses are described. A possible role \nof these viruses in human and animal health disorders is discussed. The isolated \nviruses remain unclassified so far.', "We describe an improved method for detecting deficiency of the acid hydrolase, \nalpha-1,4-glucosidase in leukocytes, the enzyme defect in glycogen storage \ndisease Type II (Pompe disease). The procedure requires smaller volumes of blood \nand less time than previous methods. The assay involves the separation of \nleukocytes by Peter's method for beta-glucosidase and a modification of Salafsky \nand Nadler's fluorometric method for alpha-glucosidase."]


In [30]:
eval_summary.to_csv(base_folder + "evaluation_results.csv", index=False)
display(eval_summary)


,Metric,Score
0,BASELINE_ROUGE-L,0.011065
1,BASELINE_BERT-F1,0.701165
2,RAG_ROUGE-L,0.038316
3,RAG_BERT-F1,0.689295
4,MAP@10,0.097645
5,MRR@10,0.221510


## Model Tuning

- Embedding model: Used `all-MiniLM-L6-v2` via HuggingFace. Consider biomedical-specific models for higher accuracy.
- Top-k: Retrieval set at 10. Can tune for different values to balance recall/precision.
- Rewrite LLM: Used DistilGPT2. Fine-tuning or using larger LLMs (e.g., Gemma, BioGPT) may improve paraphrasing for biomedical queries.
- Out-of-context filtering: Keyword-based; consider intent classification for robustness.

## Next Steps

- Experiment with more advanced out-of-context detection (intent classifier or zero-shot LLM).
- Try cross-encoder reranking after retrieval for improved final ranking.
- Incorporate feedback-based learning for continuous improvement.
- Hyperparameter tuning for k, prompt engineering, model selection.


In [31]:
def rag_chatbot(user_question):
    # Out-of-context filter
    if not is_in_context(user_question):
        return "Sorry, I can only answer biomedical science questions."
    
    # Query rewriting (using existing function for batch; here as a single-item list)
    rewritten = rewrite_queries([user_question])[0]
    
    # Retrieval (top-k passages)
    top_docs = vector_db.similarity_search(rewritten, k=10)
    top_passages = [doc.page_content for doc in top_docs]
    
    # Generation (final RAG answer)
    answer = generate_rag_answer(user_question, top_passages, tokenizer, model)
    return answer


In [33]:
# Example usage:
question = "What is the effect of the p53 mutation in cancer?"
print("User:", question)
print("Bot:", rag_chatbot(question))

User: What is the effect of the p53 mutation in cancer?


Rewriting Queries: 100%|██████████| 1/1 [00:00<00:00,  2.14it/s]


Bot: The p53 mutation is a result of a mutation in the p53 gene. The p53 mutation is a result of a mutation in the p53 gene. The p53 mutation is a result of a mutation in the p53 gene. The p53 mutation is a result of a mutation in the p53 gene. The p53 mutation is a result of a mutation in the p53 gene. The p53 mutation is a result of a mutation in the p53 gene. The p53 mutation is a result of a mutation in the p53 gene. The p53 mutation is a result of a mutation in the p53 gene.


# Tuning